# Notebook 05 — Legal RAG for grounded compliance checks
**Goal:** Ingest real German/EU legal text into Pinecone, then verify the compliance checker cites actual law instead of guessing.

By the end of this notebook you will have:
- Ingested HWG, §5a UWG, and §3a UWG into a separate Pinecone namespace (`eu-regulations`)
- Verified retrieval pulls the right section for a given claim
- Compared grounded vs ungrounded compliance verdicts on the same test claims

**Why this matters:** The compliance checker previously had two layers — a hardcoded phrase blocklist, and an LLM classifier that judged claims using GPT's general (unverified, uncitable) knowledge of "EU law". This notebook adds a third option: RAG-grounded classification, where the LLM only reasons over legal text we actually retrieved and can point back to.

## Step 1 — Environment check

In [ ]:
import sys
sys.path.append('..')

from src.utils.config import OPENAI_API_KEY, PINECONE_API_KEY, PINECONE_INDEX_NAME

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print('✅ Pinecone key loaded:', PINECONE_API_KEY[:8] + '...')
print(f'✅ Pinecone index: {PINECONE_INDEX_NAME}')

## Step 2 — Inspect the source documents
Plain-text legal source files live in `data/legal/`. Each was pulled from the official source
(gesetze-im-internet.de) and cleaned of HTML/navigation cruft, preserving § section markers.

In [ ]:
from pathlib import Path

legal_dir = Path('../data/legal')
for f in sorted(legal_dir.glob('*.txt')):
    text = f.read_text(encoding='utf-8')
    print(f'{f.name}: {len(text)} chars')
    print(f'  Preview: {text[:120]}...')
    print()

## Step 3 — Chunk one document and inspect the sections
Legal text is chunked by `§` marker, not fixed character count, so each chunk is a
complete, independently citable provision.

In [ ]:
from src.ingestion.legal_docs import chunk_legal_text

hwg_text = (legal_dir / 'hwg.txt').read_text(encoding='utf-8')
hwg_chunks = chunk_legal_text(hwg_text, law_name='HWG', source_url='https://www.gesetze-im-internet.de/heilmwerbg/')

print(f'Total HWG sections: {len(hwg_chunks)}\n')
for c in hwg_chunks[:5]:
    print(f"{c['metadata']['section']}: {c['text'][:150]}...")
    print()

## Step 4 — Ingest all legal documents into Pinecone
Embeds and upserts each document into the `eu-regulations` namespace — separate from
the video transcript corpus, so the two never mix in similarity search results.

In [ ]:
from src.ingestion.legal_docs import ingest_legal_document

documents = [
    {
        'file_path': str(legal_dir / 'hwg.txt'),
        'law_name': 'HWG',
        'source_url': 'https://www.gesetze-im-internet.de/heilmwerbg/',
    },
    {
        'file_path': str(legal_dir / 'uwg_5a.txt'),
        'law_name': 'UWG',
        'source_url': 'https://www.gesetze-im-internet.de/uwg_2004/__5a.html',
    },
    {
        'file_path': str(legal_dir / 'uwg_3a.txt'),
        'law_name': 'UWG',
        'source_url': 'https://www.gesetze-im-internet.de/uwg_2004/__3a.html',
    },
]

for doc in documents:
    summary = ingest_legal_document(**doc)
    print(f"{summary['law_name']}: {summary['section_count']} section(s), "
          f"{summary['vectors_upserted']} vectors upserted to namespace '{summary['namespace']}'")

## Step 5 — Verify the namespace in Pinecone

In [ ]:
from pinecone import Pinecone
from src.ingestion.legal_docs import LEGAL_NAMESPACE

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()

print('Namespaces in index:')
for ns, ns_stats in stats.get('namespaces', {}).items():
    label = ns if ns else '(default — video transcripts)'
    print(f'  {label}: {ns_stats["vector_count"]} vectors')

assert LEGAL_NAMESPACE in stats.get('namespaces', {}), 'Legal namespace not found — did ingestion succeed?'
print(f'\n✅ {LEGAL_NAMESPACE} namespace confirmed.')

## Step 6 — Test retrieval directly
Embed a claim and confirm the right legal section comes back.

In [ ]:
from src.compliance.checker import _retrieve_relevant_law

test_claims = [
    'This serum is clinically proven to cure acne within days.',
    'Use code SARAH10 for a discount — link in bio (no disclosure that this is paid).',
    'I love this moisturizer, my skin feels great.',
]

for claim in test_claims:
    print(f'Claim: "{claim}"')
    results = _retrieve_relevant_law(claim, k=2)
    for r in results:
        print(f"  → {r['section']} {r['law_name']} (score: {r['score']:.3f})")
        print(f"    {r['text'][:150]}...")
    print()

## Step 7 — Compare grounded vs ungrounded compliance verdicts
Same claims, run through the full `check_compliance` pipeline. Non-blocklist claims
now go through the RAG-grounded classifier and should cite specific §§.

In [ ]:
from src.compliance.checker import check_compliance

edge_cases = [
    'Use code SARAH10 for a discount — link in bio.',  # undisclosed ad, no blocklist hit
    'After using this for two weeks my eczema was completely gone and never came back.',
    'I noticed my skin felt smoother after using this moisturizer for a month.',  # should be compliant
]

for claim in edge_cases:
    result = check_compliance(claim, use_llm_fallback=True)
    status = '🚨 NON-COMPLIANT' if not result['compliant'] else '✅ COMPLIANT'
    print(f'{status} | source: {result["source"]}')
    print(f'  Claim: "{claim}"')
    print(f'  Reason: {result["reason"]}')
    if result.get('cited_sections'):
        print(f'  Cited sections: {result["cited_sections"]}')
    print()

## Step 8 — Full report output
This is what the agent tool returns to the chat UI.

In [ ]:
from src.compliance.checker import compliance_report

print(compliance_report('Use code SARAH10 for a discount — link in bio.'))
print('\n' + '='*60 + '\n')
print(compliance_report('I noticed my skin felt smoother after using this moisturizer for a month.'))

## Notes

**Corpus scope (current):** HWG (full text), §5a UWG, §3a UWG. See the README TODO for
documents still to add: EU Cosmetics Regulation 1223/2009 Art. 20, and the Cologne 2025 ruling.

**Fallback behaviour:** If `eu-regulations` is empty (e.g. fresh Pinecone index),
`check_compliance` automatically falls back to the ungrounded LLM classifier —
nothing breaks, but verdicts won't cite specific sections until this notebook is run.

**Re-running ingestion:** Safe — chunk IDs are deterministic (MD5 of law name + section),
so re-running this notebook overwrites existing vectors rather than duplicating them.

**Extending the corpus:** Add a new `.txt` file to `data/legal/`, preserve `§ N` markers
on their own line if it's a German-style law, then add an entry to the `documents` list
in Step 4 and re-run.